## SwiGLU

Every layer of an LLM has two distinct halves:

The Attention Mechanism (The Router)

The Feed-Forward Network (The Brain)

1. Attention (Horizontal Mixing)In the Attention layer, tokens look at each other.If the sentence is "The bank of the river," the word "bank" uses its Query vector to look horizontally across the sentence. It sees "river," and realizes it means a muddy shore, not a financial institution.Attention mixes information across the sequence length ($T$). However, it doesn't actually add new fundamental knowledge to the model. It just routes the context that is already there.2. The FFN (Vertical Mixing)Once "bank" realizes it means "muddy shore," it exits the Attention layer and enters the FFN.Crucially, the FFN does not look at other words. In the FFN, the token "bank" is entirely isolated. The network only mixes information across the embedding dimension ($d_{model}$) of that single, specific word.Think of the FFN as a massive database of memorized facts and logic circuits.The token comes in saying: "I am the concept of a muddy river bank in the context of nature."It passes through the FFN's massive hidden layers.The FFN acts as a lookup table. It fires specific neurons that associate "river bank" with concepts like "water," "erosion," "frogs," and "mud."The token exits the FFN significantly heavier, enriched with actual world knowledge.Attention gives the word its context. The FFN gives the word its definition and facts. You cannot have intelligence without both.

# New gate instead of ReLU
The New Way: Gating Mechanisms (The Scalpel)AI researchers realized that to make LLMs smarter, they needed a smoother, more intelligent way to filter knowledge. They needed a Gate.Instead of a harsh "yes or no" cutoff, a Gate operates on probabilities.Imagine two streams of data flowing through the FFN simultaneously:The Knowledge Stream: The actual facts the network is retrieving.The Gate Stream: A separate neural layer whose entire job is to calculate a value between $0.0$ and $1.0$.The network physically multiplies the Knowledge Stream by the Gate Stream.If the Gate outputs $0.9$, 90% of the knowledge passes through.If the Gate outputs $0.1$, only 10% passes through.Why is this brilliant?The network is no longer using a hard-coded math rule (like ReLU) to destroy data. It is using a learned neural network to dynamically decide exactly how much of a specific fact is relevant to the current word. It is smooth, continuous, and preserves the intricate gradients of language.This gating philosophy is what gave birth to SwiGLU, the specific architecture that makes Llama 3 and DeepSeek so powerful.

1. The "Swi" (Swish / SiLU Activation)Researchers at Google Brain used automated search algorithms to find the mathematical curve that makes neural networks learn fastest. They discovered Swish (specifically, the variant called SiLU - Sigmoid Linear Unit).Here is the formula for SiLU:$$f(x) = x \cdot \sigma(x)$$(Where $\sigma(x)$ is the Sigmoid function, which squishes any number into a probability between 0 and 1).Why is SiLU better than ReLU?No Dead Neurons: Remember how ReLU violently crushes all negative numbers to exactly $0$? SiLU allows a tiny bit of negative information to leak through. It creates a smooth, gentle dip below zero before flattening out.Smooth Gradients: Because the curve is smooth (no sharp angles like ReLU), the calculus used during training flows much easier, allowing the LLM to learn subtle, complex relationships in language.2. The "GLU" (Gated Linear Unit)Now we add the Gate.In a standard, old-school FFN, you take your word $x$, multiply it by a weight matrix $W_1$, apply an activation function, and then multiply by $W_2$:$$FFN_{Standard}(x) = \text{ReLU}(x \cdot W_1) \cdot W_2$$A GLU splits the first step into two completely separate parallel paths. It creates two completely different matrices ($W_{Gate}$ and $W_{Up}$).The Gate Path: $x \cdot W_{Gate}$ (This passes through the SiLU curve to become a probability filter).The Up Path (The Knowledge): $x \cdot W_{Up}$ (This is the raw data, completely unfiltered).Then, it physically multiplies them together element-by-element (denoted by the $\otimes$ symbol):$$\text{SwiGLU\_Output} = (\text{SiLU}(x \cdot W_{Gate}) \otimes (x \cdot W_{Up}))$$Finally, we pass that filtered result through a final projection matrix ($W_{Down}$) to shrink it back to the normal token size.The Ultimate SwiGLU Equation:$$FFN_{SwiGLU}(x) = (\text{SiLU}(x \cdot W_{Gate}) \otimes (x \cdot W_{Up})) \cdot W_{Down}$$Instead of a dumb axe (ReLU) chopping off negative data, SwiGLU has a dedicated neural network ($W_{Gate}$) deciding exactly which facts from the knowledge stream ($W_{Up}$) are allowed to pass through the filter.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LlamaSwiGLU(nn.Module):
    def __init__(self, d_model=1024):
        super().__init__()
        self.d_model = d_model
        
        # --- THE DIMENSION HACK ---
        # Standard FFN expands hidden dimension by 4x.
        # SwiGLU expands by 8/3 (approx 2.66x) to compensate for having 3 matrices instead of 2.
        hidden_dim = int(4 * d_model * (2/3)) # 4 * 1024 * (2/3) = 2730.66
        
        # GPUs hate weird numbers. They love multiples of 256 for optimal hardware math.
        # So we round 2730 up to the nearest multiple of 256.
        multiple_of = 256
        self.hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of) # Rounds to 2816
        
        # --- THE THREE MATRICES ---
        self.w_gate = nn.Linear(self.d_model, self.hidden_dim, bias=False)
        self.w_up   = nn.Linear(self.d_model, self.hidden_dim, bias=False)
        self.w_down = nn.Linear(self.hidden_dim, self.d_model, bias=False)

        self.print_setup()

    def print_setup(self):
        print("="*80)
        print(f"{'SwiGLU FFN INITIALIZATION':^80}")
        print("="*80)
        print(f"Input Dimension:   {self.d_model}")
        print(f"Hidden Dimension:  {self.hidden_dim} (Rounded for GPU efficiency)")
        print(f"Matrix Weights:    w_gate, w_up, w_down (No biases used in Llama 3!)")
        print("-" * 80)

    def forward(self, x):
        B, T, C = x.shape
        print(f"[Step 0] Input Shape:         {list(x.shape)} -> (Batch, Tokens, Dim)")
        
        # --- 1. The Gate Stream (The Filter) ---
        # We pass the data through w_gate, then apply the smooth SiLU activation.
        # This creates our probability filter.
        raw_gate = self.w_gate(x)
        gate_stream = F.silu(raw_gate)
        print(f"[Step 1] Gate Stream:         {list(gate_stream.shape)} -> SiLU applied")
        
        # --- 2. The Knowledge Stream (The Raw Facts) ---
        # We pass the exact same input through a completely different matrix.
        # This is the raw, unfiltered knowledge. NO activation function is used here.
        knowledge_stream = self.w_up(x)
        print(f"[Step 2] Knowledge Stream:    {list(knowledge_stream.shape)} -> Raw data")
        
        # --- 3. The Multiplication (The Magic) ---
        # We physically multiply the two streams together element-by-element.
        # The gate dictates exactly which facts from the knowledge stream survive.
        filtered_knowledge = gate_stream * knowledge_stream
        print(f"[Step 3] Filtered Knowledge:  {list(filtered_knowledge.shape)} -> Element-wise multiply")
        
        # --- 4. The Compression (Down Projection) ---
        # We shrink the massive 2816-dimensional filtered facts back down to 1024 
        # so it can continue to the next layer of the Transformer.
        out = self.w_down(filtered_knowledge)
        print(f"[Step 4] Final Output Shape:  {list(out.shape)} -> Compressed back to d_model")
        print("="*80)
        
        return out

if __name__ == "__main__":
    torch.manual_seed(42)
    dummy_input = torch.randn(1, 10, 1024) # Batch=1, Tokens=10, d_model=1024
    ffn = LlamaSwiGLU(d_model=1024)
    output = ffn(dummy_input)
    print("✅ SwiGLU Block Executed Successfully!")

                           SwiGLU FFN INITIALIZATION                            
Input Dimension:   1024
Hidden Dimension:  2816 (Rounded for GPU efficiency)
Matrix Weights:    w_gate, w_up, w_down (No biases used in Llama 3!)
--------------------------------------------------------------------------------
[Step 0] Input Shape:         [1, 10, 1024] -> (Batch, Tokens, Dim)
[Step 1] Gate Stream:         [1, 10, 2816] -> SiLU applied
[Step 2] Knowledge Stream:    [1, 10, 2816] -> Raw data
[Step 3] Filtered Knowledge:  [1, 10, 2816] -> Element-wise multiply
[Step 4] Final Output Shape:  [1, 10, 1024] -> Compressed back to d_model
✅ SwiGLU Block Executed Successfully!
